In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q -U bitsandbytes accelerate peft transformers datasets torch tqdm

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model

# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
DATASET_TXT_PATH = "/content/drive/MyDrive/LoRA_FlexiLM/medical_clean_final.txt"
OUTPUT_DIR = "./lora_adapter_qwen05b_medical"

NUM_EPOCHS = 3
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
LR = 1e-4
MAX_LENGTH = 512

# =========================================================
# LOAD MODEL
# =========================================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded\n")


# =========================================================
# DATASET
# =========================================================

def prepare_dataset(txt_path):

    dataset = load_dataset("text", data_files={"train": txt_path})

    def tokenize(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False
        )

    tokenized = dataset.map(
        tokenize,
        batched=True,
        remove_columns=["text"]
    )

    return tokenized["train"]


train_dataset = prepare_dataset(DATASET_TXT_PATH)

print(f"Dataset size: {len(train_dataset)}")


# =========================================================
# LORA CONFIG
# =========================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


# =========================================================
# TRAINING CONFIG
# =========================================================

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=LR,

    fp16=True,

    logging_steps=50,

    save_strategy="epoch",

    report_to="none",

    optim="adamw_8bit",

    lr_scheduler_type="cosine",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


# =========================================================
# TRAINER
# =========================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    data_collator=data_collator
)


# =========================================================
# TRAIN
# =========================================================

print("\nStarting fine-tuning...\n")

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining complete.")
print(f"Adapter saved at: {OUTPUT_DIR}")


# =========================================================
# TOKEN ACCURACY CALCULATION
# =========================================================

print("\nCalculating token accuracy...\n")

model.eval()

correct = 0
total = 0

sample_size = min(300, len(train_dataset))

for sample in train_dataset.select(range(sample_size)):

    input_ids = torch.tensor(sample["input_ids"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        outputs = model(input_ids)

    logits = outputs.logits

    # shift for next-token prediction
    preds = torch.argmax(logits[:, :-1], dim=-1)
    labels = input_ids[:, 1:]

    correct += (preds == labels).sum().item()
    total += labels.numel()

accuracy = (correct / total) * 100


print("\n" + "="*55)
print(f"TRAIN TOKEN ACCURACY: {accuracy:.2f}%")
print("="*55)

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded

Dataset size: 8084
trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359

Starting fine-tuning...



Step,Training Loss
50,1.452156
100,1.047818
150,1.019719
200,1.001718
250,0.952860
300,0.961618
350,0.955191
400,0.936783
450,0.941099
500,0.925132



Training complete.
Adapter saved at: ./lora_adapter_qwen05b_medical

Calculating token accuracy...


TRAIN TOKEN ACCURACY: 82.39%


**1st**

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import json
import os
import zipfile
import tempfile
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model

class StreamlitProgressCallback(TrainerCallback):
    def __init__(self, progress_bar, text_container, total_epochs):
        self.progress_bar = progress_bar
        self.text_container = text_container
        self.total_epochs = total_epochs
        self.current_epoch = 0

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.current_epoch += 1
        progress = self.current_epoch / self.total_epochs
        self.progress_bar.progress(progress)
        self.text_container.markdown(
            f'<div style="text-align:center; font-weight:600; color:#1e40af; margin-bottom:8px;">'
            f'Epoch {self.current_epoch}/{self.total_epochs} • {int(progress*100)}%</div>',
            unsafe_allow_html=True
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            self.text_container.markdown(
                f'<div style="text-align:center; font-weight:600; color:#1e40af; margin-bottom:8px;">'
                f'Epoch {self.current_epoch}/{self.total_epochs} • '
                f'loss: {logs["loss"]:.4f}</div>',
                unsafe_allow_html=True
            )

st.set_page_config(
    page_title="No-Code LoRA Fine-Tuning",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="collapsed"
)

st.markdown("""
<style>
    /* Force light theme */
    .stApp, .stApp > header, .stApp > div {
        background: linear-gradient(135deg, #f5f7fa 0%, #e9ecf5 100%) !important;
        color: #1e293b !important;
    }
    [data-testid="stAppViewContainer"] {
        background: linear-gradient(135deg, #f5f7fa 0%, #e9ecf5 100%) !important;
    }

    /* Kill excessive top/bottom padding */
    .block-container {
        padding-top: 1.2rem !important;
        padding-bottom: 1rem !important;
    }
    section[data-testid="stTabs"] > div:first-child {
        margin-bottom: 0 !important;
        padding-bottom: 0 !important;
    }
    .stTabs [data-baseweb="tab-panel"] {
        padding-top: 0.5rem !important;
        margin-top: -0.4rem !important;
    }
    div[data-testid="stDecoration"] {
        display: none !important;
    }

    /* Main title */
    .main-title {
        background: linear-gradient(90deg, #1e3a8a, #3b82f6);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-size: 3.1rem !important;
        font-weight: 700 !important;
        text-align: center;
        margin: 0.3rem 0 0.1rem 0 !important;
    }
    .sub-title {
        text-align: center;
        color: #2c3e50;
        font-size: 1.22rem;
        margin: 0 0 1.1rem 0;
    }

    /* Cards */
    .card {
        background: white;
        border-radius: 18px;
        padding: 1.5rem 1.8rem;
        box-shadow: 0 8px 20px -4px rgba(0,0,0,0.08);
        margin-bottom: 1.1rem;
        border: 1px solid rgba(59,130,246,0.07);
    }
    .card:last-child {
        margin-bottom: 0.6rem !important;
    }

    /* Progress area */
    .progress-container {
        background: white;
        border-radius: 14px;
        padding: 1.3rem;
        box-shadow: 0 3px 12px rgba(0,0,0,0.05);
        margin: 0.8rem 0 1.4rem 0;
    }

    /* Buttons, inputs, etc. */
    .stButton > button {
        background: linear-gradient(90deg, #1e3a8a, #2563eb);
        color: white;
        border: none;
        border-radius: 40px;
        padding: 0.6rem 2rem;
        font-weight: 600;
        box-shadow: 0 4px 6px -1px rgba(37,99,235,0.25);
    }
    .stButton > button:hover {
        background: linear-gradient(90deg, #1e3a8a, #1d4ed8);
        transform: translateY(-1px);
    }
    .stFileUploader > div:first-child {
        border: 2px dashed #3b82f6;
        border-radius: 28px;
        background: rgba(59,130,246,0.03);
        padding: 1.8rem;
    }
</style>
""", unsafe_allow_html=True)

if 'finetuned_model' not in st.session_state:
    st.session_state.finetuned_model = None
if 'finetuned_tokenizer' not in st.session_state:
    st.session_state.finetuned_tokenizer = None
if 'adapter_path' not in st.session_state:
    st.session_state.adapter_path = None
if 'chat_history' not in st.session_state:
    st.session_state.chat_history = []

MODEL_OPTIONS = {
    "Qwen 0.5B Instruct": "Qwen/Qwen2-0.5B-Instruct",
    "Gemma 2B": "google/gemma-2b",
    "OPT 1.3B": "facebook/opt-1.3b",
    "Pythia 1.4B": "EleutherAI/pythia-1.4b-deduped",
}

REQUIRES_TOKEN = {
    "Qwen 0.5B Instruct": False,
    "Gemma 2B": True,
    "OPT 1.3B": False,
    "Pythia 1.4B": False,
}

def prepare_dataset(file_path, tokenizer):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if len(line.strip()) >= 30]
    if not lines:
        raise ValueError("No valid lines found (minimum 30 characters per line)")

    formatted = []
    for text in lines:
        formatted.append(
            f"User: What does this text explain?\n\n"
            f"Assistant: {text}<|im_end|>\n"
        )

    jsonl_path = tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False)
    for example in formatted:
        jsonl_path.write(json.dumps({"text": example}, ensure_ascii=False) + "\n")
    jsonl_path.close()

    dataset = load_dataset("json", data_files=jsonl_path.name, split="train")

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=512,
            padding=False
        )

    train_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    os.unlink(jsonl_path.name)
    return train_dataset

def load_base_model_for_lora(model_name, hf_token=None):
    model_id = MODEL_OPTIONS[model_name]
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, token=hf_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        token=hf_token
    )

    if "Qwen" in model_name or "Qwen" in model_id:
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
    elif "gemma" in model_id.lower():
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
    elif "opt" in model_id.lower():
        target_modules = ["q_proj", "k_proj", "v_proj", "out_proj"]
    elif "pythia" in model_id.lower():
        target_modules = ["q_proj", "k_proj", "v_proj", "dense"]
    else:
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer

def start_training(model_name, dataset_file, hf_token):
    try:
        if REQUIRES_TOKEN[model_name] and not hf_token.strip():
            st.error(f"{model_name} requires a Hugging Face access token.")
            return False

        progress_container = st.empty()
        with progress_container.container():
            st.markdown('<div class="progress-container">', unsafe_allow_html=True)
            progress_text = st.empty()
            progress_bar = st.progress(0)
            st.markdown('</div>', unsafe_allow_html=True)

        with st.status("Preparing model & data...", expanded=True) as status:
            model, tokenizer = load_base_model_for_lora(model_name, hf_token or None)
            status.update(label="Tokenizing dataset...", state="running")
            train_dataset = prepare_dataset(dataset_file, tokenizer)

            status.update(label="Setting up LoRA + Trainer...", state="running")
            adapter_path = f"./lora_adapter_{model_name.replace(' ', '_')}"
            os.makedirs(adapter_path, exist_ok=True)

            training_args = TrainingArguments(
                output_dir=adapter_path,
                num_train_epochs=3,
                per_device_train_batch_size=8,
                gradient_accumulation_steps=4,
                learning_rate=1.5e-4,
                fp16=True,
                logging_steps=20,
                save_strategy="epoch",
                report_to="none",
                optim="adamw_torch",
                lr_scheduler_type="cosine",
                warmup_ratio=0.05,
            )

            progress_callback = StreamlitProgressCallback(
                progress_bar=progress_bar,
                text_container=progress_text,
                total_epochs=training_args.num_train_epochs
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=train_dataset,
                data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
                callbacks=[progress_callback],
            )

            status.update(label="Fine-tuning started...", state="running")
            trainer.train()

            status.update(label="Saving adapter...", state="running")
            model.save_pretrained(adapter_path)
            tokenizer.save_pretrained(adapter_path)

            st.session_state.finetuned_model = model
            st.session_state.finetuned_tokenizer = tokenizer
            st.session_state.adapter_path = adapter_path

            status.update(label="Training finished!", state="complete")

        progress_container.empty()
        st.success(f"✅ LoRA adapter saved → {adapter_path}")
        return True

    except Exception as e:
        st.error(f"Training failed: {str(e)}")
        return False

def chat_response(message):
    if st.session_state.finetuned_model is None:
        return "Please train a model first."

    prompt = f"User: {message}\n\nAssistant:"
    inputs = st.session_state.finetuned_tokenizer(prompt, return_tensors="pt").to(st.session_state.finetuned_model.device)

    possible_eos = [
        st.session_state.finetuned_tokenizer.eos_token_id,
        st.session_state.finetuned_tokenizer.convert_tokens_to_ids("<|im_end|>"),
        st.session_state.finetuned_tokenizer.convert_tokens_to_ids("<|endoftext|>"),
    ]
    eos_ids = [eid for eid in possible_eos if eid is not None]

    with torch.inference_mode():
        outputs = st.session_state.finetuned_model.generate(
            **inputs,
            max_new_tokens=220,
            temperature=0.45,
            top_p=0.92,
            do_sample=True,
            repetition_penalty=1.18,
            eos_token_id=eos_ids,
            pad_token_id=st.session_state.finetuned_tokenizer.eos_token_id,
        )

    full_text = st.session_state.finetuned_tokenizer.decode(outputs[0], skip_special_tokens=False)
    if "Assistant:" in full_text:
        answer = full_text.split("Assistant:", 1)[1].strip()
    else:
        answer = full_text[len(prompt):].strip()

    for junk in ["<|im_end|>", "<|endoftext|>", "<|im_end]>", "</s>", "<s>"]:
        answer = answer.replace(junk, "").strip()

    answer = answer.split("User:")[0].strip()
    return answer or "(no clear response — try asking differently)"

def export_adapter():
    if not st.session_state.adapter_path or not os.path.exists(st.session_state.adapter_path):
        st.warning("No trained adapter found. Please train first.")
        return None

    zip_path = st.session_state.adapter_path + ".zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(st.session_state.adapter_path):
            for file in files:
                full = os.path.join(root, file)
                arc = os.path.relpath(full, st.session_state.adapter_path)
                zf.write(full, arc)
    return zip_path

st.markdown('<h1 class="main-title">🧠 No-Code LoRA Fine-Tuning</h1>', unsafe_allow_html=True)
st.markdown('<p class="sub-title">Upload .txt → Choose model → Train → Chat → Download adapter</p>', unsafe_allow_html=True)

tab1, tab2, tab3 = st.tabs(["🚀 Train", "💬 Chat", "📥 Export"])

with tab1:
    with st.container():
        st.markdown('<div class="card">', unsafe_allow_html=True)
        st.subheader("Train your model")

        col1, col2 = st.columns(2)
        with col1:
            model_name = st.selectbox("Base model", list(MODEL_OPTIONS.keys()), index=0)
        with col2:
            dataset_file = st.file_uploader("Upload your dataset (.txt)", type="txt")

        hf_token = st.text_input(
            "Hugging Face Token (only needed for Gemma)",
            type="password",
            placeholder="hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
            help="Required only if you choose Gemma 2B"
        )

        if st.button("🚀 Start Fine-Tuning", type="primary", use_container_width=True):
            if dataset_file is None:
                st.warning("Please upload a .txt dataset file.")
            else:
                with tempfile.NamedTemporaryFile(delete=False, suffix=".txt") as temp_file:
                    temp_file.write(dataset_file.getvalue())
                    temp_path = temp_file.name

                success = start_training(model_name, temp_path, hf_token)
                os.unlink(temp_path)

                if success:
                    st.balloons()

        st.markdown('</div>', unsafe_allow_html=True)

with tab2:
    with st.container():
        st.markdown('<div class="card">', unsafe_allow_html=True)
        st.subheader("Chat with your fine-tuned model")

        if st.session_state.finetuned_model is None:
            st.info("👋 Please train a model first in the Train tab.")
        else:
            for msg in st.session_state.chat_history:
                with st.chat_message(msg["role"]):
                    st.markdown(msg["content"])

            user_input = st.chat_input("Ask a question related to your data...")
            if user_input:
                st.session_state.chat_history.append({"role": "user", "content": user_input})
                with st.chat_message("user"):
                    st.markdown(user_input)

                with st.chat_message("assistant"):
                    with st.spinner("Thinking..."):
                        response = chat_response(user_input)
                    st.markdown(response)

                st.session_state.chat_history.append({"role": "assistant", "content": response})

        if st.button("🧹 Clear conversation", use_container_width=True):
            st.session_state.chat_history = []
            st.rerun()

        st.markdown('</div>', unsafe_allow_html=True)
with tab3:
    with st.container():
        st.markdown('<div class="card">', unsafe_allow_html=True)
        st.subheader("Export your trained adapter")

        if st.session_state.adapter_path:
            zip_path = export_adapter()
            if zip_path:
                with open(zip_path, "rb") as f:
                    st.download_button(
                        label="📥 Download Adapter (.zip)",
                        data=f,
                        file_name="lora_adapter.zip",
                        mime="application/zip",
                        use_container_width=True
                    )
        else:
            st.info("Train a model first to generate an adapter.")

        st.markdown('</div>', unsafe_allow_html=True)

st.markdown("""
<div style="text-align:center; color:#6b7280; font-size:0.9rem; margin:2.5rem 0 1rem 0; padding-top:1.2rem; border-top:1px solid #e2e8f0;">
    Training speed depends on dataset size & GPU availability.<br>
    Small models like Qwen 0.5B are fastest and use least memory.
</div>
""", unsafe_allow_html=True)

Writing app.py


**2ND**

In [ ]:

NGROK_AUTH_TOKEN = "3AFlg9Z9NHzr6veHUCHSuNQzSDV_6GGZ6HYQpdmy9cNrLam36"

!pip install streamlit pyngrok -q

from pyngrok import ngrok
import os
import time

os.system("pkill -f streamlit")
os.system("pkill -f ngrok")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

get_ipython().system_raw(
    "streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true &"
)

time.sleep(6)

public_url = ngrok.connect(8501)

print("Streamlit App Running:")
print(public_url)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 40.3 MB/s eta 0:00:00
🚀 Streamlit App Running:
NgrokTunnel: "https://unmilitaristic-alexander-intermetallic.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
# ngrok + Streamlit launc)

NGROK_AUTH_TOKEN = "3AFlg9Z9NHzr6veHUCHSuNQzSDV_6GGZ6HYQpdmy9cNrLam36"  # https://dashboard.ngrok.com/get-started/your-authtoken

!pip install pyngrok --quiet
from pyngrok import ngrok
import time
import os

os.system("pkill -f ngrok || true")

os.system(f"ngrok authtoken {NGROK_AUTH_TOKEN}")

public_url = ngrok.connect(8501, "http")
print(f"Public Streamlit URL: {public_url}")

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --theme.base light &>/dev/null &

Public Streamlit URL: NgrokTunnel: "https://unmilitaristic-alexander-intermetallic.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
%%writefile app.py
import streamlit as st
import torch
import json
import os
import zipfile
import tempfile
import math
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model

# ────────────────────────────────────────────────
#  Custom callback – shows epoch & loss progress
# ────────────────────────────────────────────────
class StreamlitProgressCallback(TrainerCallback):
    def __init__(self, progress_bar, text_container, total_epochs):
        self.progress_bar = progress_bar
        self.text_container = text_container
        self.total_epochs = total_epochs
        self.current_epoch = 0

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.current_epoch += 1
        progress = self.current_epoch / self.total_epochs
        self.progress_bar.progress(progress)
        self.text_container.markdown(
            f'<div style="text-align:center; font-weight:600; color:#1e40af; margin-bottom:8px;">'
            f'Epoch {self.current_epoch}/{self.total_epochs} • {int(progress*100)}%</div>',
            unsafe_allow_html=True
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            self.text_container.markdown(
                f'<div style="text-align:center; font-weight:600; color:#1e40af; margin-bottom:8px;">'
                f'Epoch {self.current_epoch}/{self.total_epochs} • loss: {logs["loss"]:.4f}</div>',
                unsafe_allow_html=True
            )

# ────────────────────────────────────────────────
#  Page config & forced light theme + layout fixes
# ────────────────────────────────────────────────
st.set_page_config(
    page_title="No-Code LoRA Fine-Tuning",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="collapsed"
)

st.markdown("""
<style>
    /* Force light theme */
    .stApp, .stApp > header, .stApp > div {
        background: linear-gradient(135deg, #f5f7fa 0%, #e9ecf5 100%) !important;
        color: #1e293b !important;
    }
    [data-testid="stAppViewContainer"] {
        background: linear-gradient(135deg, #f5f7fa 0%, #e9ecf5 100%) !important;
    }

    /* Reduce excessive spacing */
    .block-container {
        padding-top: 1.2rem !important;
        padding-bottom: 1rem !important;
    }
    section[data-testid="stTabs"] > div:first-child {
        margin-bottom: 0 !important;
        padding-bottom: 0 !important;
    }
    .stTabs [data-baseweb="tab-panel"] {
        padding-top: 0.5rem !important;
        margin-top: -0.4rem !important;
    }
    div[data-testid="stDecoration"] {
        display: none !important;
    }

    /* Titles */
    .main-title {
        background: linear-gradient(90deg, #1e3a8a, #3b82f6);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-size: 3.1rem !important;
        font-weight: 700 !important;
        text-align: center;
        margin: 0.3rem 0 0.1rem 0 !important;
    }
    .sub-title {
        text-align: center;
        color: #2c3e50;
        font-size: 1.22rem;
        margin: 0 0 1.1rem 0;
    }

    /* Cards */
    .card {
        background: white;
        border-radius: 18px;
        padding: 1.5rem 1.8rem;
        box-shadow: 0 8px 20px -4px rgba(0,0,0,0.08);
        margin-bottom: 1.1rem;
        border: 1px solid rgba(59,130,246,0.07);
    }
    .card:last-child {
        margin-bottom: 0.6rem !important;
    }

    /* Progress container */
    .progress-container {
        background: white;
        border-radius: 14px;
        padding: 1.3rem;
        box-shadow: 0 3px 12px rgba(0,0,0,0.05);
        margin: 0.8rem 0 1.4rem 0;
    }

    /* Result box after training */
    .result-box {
        background: #f0f9ff;
        border-radius: 12px;
        padding: 1.4rem;
        margin: 1.2rem 0;
        border-left: 5px solid #3b82f6;
    }

    /* Buttons & uploader */
    .stButton > button {
        background: linear-gradient(90deg, #1e3a8a, #2563eb);
        color: white;
        border: none;
        border-radius: 40px;
        padding: 0.6rem 2rem;
        font-weight: 600;
        box-shadow: 0 4px 6px -1px rgba(37,99,235,0.25);
    }
    .stButton > button:hover {
        background: linear-gradient(90deg, #1e3a8a, #1d4ed8);
        transform: translateY(-1px);
    }
    .stFileUploader > div:first-child {
        border: 2px dashed #3b82f6;
        border-radius: 28px;
        background: rgba(59,130,246,0.03);
        padding: 1.8rem;
    }
</style>
""", unsafe_allow_html=True)

# ────────────────────────────────────────────────
#  Session state
# ────────────────────────────────────────────────
if 'finetuned_model' not in st.session_state:
    st.session_state.finetuned_model = None
if 'finetuned_tokenizer' not in st.session_state:
    st.session_state.finetuned_tokenizer = None
if 'adapter_path' not in st.session_state:
    st.session_state.adapter_path = None
if 'chat_history' not in st.session_state:
    st.session_state.chat_history = []

# ────────────────────────────────────────────────
#  Model registry
# ────────────────────────────────────────────────
MODEL_OPTIONS = {
    "Qwen 0.5B Instruct": "Qwen/Qwen2-0.5B-Instruct",
    "Gemma 2B": "google/gemma-2b",
    "OPT 1.3B": "facebook/opt-1.3b",
    "Pythia 1.4B": "EleutherAI/pythia-1.4b-deduped",
}

REQUIRES_TOKEN = {
    "Qwen 0.5B Instruct": False,
    "Gemma 2B": True,
    "OPT 1.3B": False,
    "Pythia 1.4B": False,
}

# ────────────────────────────────────────────────
#  Dataset → tokenized train + optional eval split
# ────────────────────────────────────────────────
def prepare_dataset(file_path, tokenizer, validation_split=0.15):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if len(line.strip()) >= 30]

    if not lines:
        raise ValueError("No valid lines found (minimum 30 characters per line)")

    formatted = []
    for text in lines:
        formatted.append(
            f"User: What does this text explain?\n\n"
            f"Assistant: {text}<|im_end|>\n"
        )

    jsonl_path = tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False)
    for example in formatted:
        jsonl_path.write(json.dumps({"text": example}, ensure_ascii=False) + "\n")
    jsonl_path.close()

    dataset = load_dataset("json", data_files=jsonl_path.name, split="train")

    # Split only if dataset is reasonably large
    if validation_split > 0 and len(dataset) > 400:
        dataset = dataset.train_test_split(test_size=validation_split, seed=42)
        train_ds = dataset["train"]
        eval_ds  = dataset["test"]
    else:
        train_ds = dataset
        eval_ds  = None

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=512,
            padding=False
        )

    tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

    tokenized_eval = None
    if eval_ds is not None:
        tokenized_eval = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

    os.unlink(jsonl_path.name)

    return tokenized_train, tokenized_eval

# ────────────────────────────────────────────────
#  Load model + apply LoRA config
# ────────────────────────────────────────────────
def load_base_model_for_lora(model_name, hf_token=None):
    model_id = MODEL_OPTIONS[model_name]
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, token=hf_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        token=hf_token
    )

    if "Qwen" in model_name or "Qwen" in model_id:
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
    elif "gemma" in model_id.lower():
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
    elif "opt" in model_id.lower():
        target_modules = ["q_proj", "k_proj", "v_proj", "out_proj"]
    elif "pythia" in model_id.lower():
        target_modules = ["q_proj", "k_proj", "v_proj", "dense"]
    else:
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer

# ────────────────────────────────────────────────
#  Training function – now computes perplexity
# ────────────────────────────────────────────────
def start_training(model_name, dataset_file, hf_token):
    try:
        if REQUIRES_TOKEN[model_name] and not hf_token.strip():
            st.error(f"{model_name} requires a Hugging Face access token.")
            return False, None

        # Progress UI
        progress_container = st.empty()
        with progress_container.container():
            st.markdown('<div class="progress-container">', unsafe_allow_html=True)
            progress_text = st.empty()
            progress_bar = st.progress(0)
            st.markdown('</div>', unsafe_allow_html=True)

        with st.status("Preparing model & data...", expanded=True) as status:
            model, tokenizer = load_base_model_for_lora(model_name, hf_token or None)
            status.update(label="Tokenizing dataset...", state="running")

            train_dataset, eval_dataset = prepare_dataset(dataset_file, tokenizer, validation_split=0.15)

            status.update(label="Setting up LoRA + Trainer...", state="running")
            adapter_path = f"./lora_adapter_{model_name.replace(' ', '_')}"
            os.makedirs(adapter_path, exist_ok=True)

            training_args = TrainingArguments(
                output_dir=adapter_path,
                num_train_epochs=3,
                per_device_train_batch_size=8,
                gradient_accumulation_steps=4,
                learning_rate=1.5e-4,
                fp16=True,
                logging_steps=20,
                save_strategy="epoch",
                report_to="none",
                optim="adamw_torch",
                lr_scheduler_type="cosine",
                warmup_ratio=0.05,
                evaluation_strategy="no" if eval_dataset is None else "epoch",
            )

            progress_callback = StreamlitProgressCallback(
                progress_bar=progress_bar,
                text_container=progress_text,
                total_epochs=training_args.num_train_epochs
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=eval_dataset,
                data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
                callbacks=[progress_callback],
            )

            status.update(label="Fine-tuning started...", state="running")
            train_result = trainer.train()

            status.update(label="Saving adapter...", state="running")
            model.save_pretrained(adapter_path)
            tokenizer.save_pretrained(adapter_path)

            st.session_state.finetuned_model = model
            st.session_state.finetuned_tokenizer = tokenizer
            st.session_state.adapter_path = adapter_path

            status.update(label="Training finished!", state="complete")

        progress_container.empty()

        # ── Calculate perplexity ───────────────────────────────
        final_train_loss = None
        if hasattr(train_result, "metrics") and "train_loss" in train_result.metrics:
            final_train_loss = train_result.metrics["train_loss"]
        elif hasattr(train_result, "log_history") and train_result.log_history:
            final_train_loss = train_result.log_history[-1].get("loss")

        train_ppl = math.exp(final_train_loss) if final_train_loss is not None else float("nan")

        eval_ppl = None
        if eval_dataset is not None and "eval_loss" in train_result.metrics:
            eval_ppl = math.exp(train_result.metrics["eval_loss"])

        metrics = {
            "train_perplexity": train_ppl,
            "eval_perplexity": eval_ppl
        }

        st.success(f"✅ LoRA adapter saved → {adapter_path}")

        # Show results
        with st.container():
            st.markdown('<div class="result-box">', unsafe_allow_html=True)
            cols = st.columns(2 if eval_ppl is not None else 1)
            with cols[0]:
                st.metric(
                    "Training Perplexity",
                    f"{train_ppl:.2f}" if not math.isnan(train_ppl) else "—",
                    help="Lower is better. Based on final average training loss."
                )
            if eval_ppl is not None:
                with cols[1]:
                    st.metric(
                        "Validation Perplexity",
                        f"{eval_ppl:.2f}",
                        help="Lower is better — more reliable quality indicator."
                    )
            st.markdown('</div>', unsafe_allow_html=True)

        return True, metrics

    except Exception as e:
        st.error(f"Training failed: {str(e)}")
        return False, None

# ────────────────────────────────────────────────
#  Chat inference
# ────────────────────────────────────────────────
def chat_response(message):
    if st.session_state.finetuned_model is None:
        return "Please train a model first."

    prompt = f"User: {message}\n\nAssistant:"
    inputs = st.session_state.finetuned_tokenizer(prompt, return_tensors="pt").to(st.session_state.finetuned_model.device)

    possible_eos = [
        st.session_state.finetuned_tokenizer.eos_token_id,
        st.session_state.finetuned_tokenizer.convert_tokens_to_ids("<|im_end|>"),
        st.session_state.finetuned_tokenizer.convert_tokens_to_ids("<|endoftext|>"),
    ]
    eos_ids = [eid for eid in possible_eos if eid is not None]

    with torch.inference_mode():
        outputs = st.session_state.finetuned_model.generate(
            **inputs,
            max_new_tokens=220,
            temperature=0.45,
            top_p=0.92,
            do_sample=True,
            repetition_penalty=1.18,
            eos_token_id=eos_ids,
            pad_token_id=st.session_state.finetuned_tokenizer.eos_token_id,
        )

    full_text = st.session_state.finetuned_tokenizer.decode(outputs[0], skip_special_tokens=False)
    if "Assistant:" in full_text:
        answer = full_text.split("Assistant:", 1)[1].strip()
    else:
        answer = full_text[len(prompt):].strip()

    for junk in ["<|im_end|>", "<|endoftext|>", "<|im_end]>", "</s>", "<s>"]:
        answer = answer.replace(junk, "").strip()

    answer = answer.split("User:")[0].strip()
    return answer or "(no clear response — try asking differently)"

# ────────────────────────────────────────────────
#  Export LoRA adapter as zip
# ────────────────────────────────────────────────
def export_adapter():
    if not st.session_state.adapter_path or not os.path.exists(st.session_state.adapter_path):
        st.warning("No trained adapter found. Please train first.")
        return None

    zip_path = st.session_state.adapter_path + ".zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(st.session_state.adapter_path):
            for file in files:
                full = os.path.join(root, file)
                arc = os.path.relpath(full, st.session_state.adapter_path)
                zf.write(full, arc)
    return zip_path

# ────────────────────────────────────────────────
#  UI – Header + Tabs
# ────────────────────────────────────────────────
st.markdown('<h1 class="main-title">🧠 No-Code LoRA Fine-Tuning</h1>', unsafe_allow_html=True)
st.markdown('<p class="sub-title">Upload .txt → Choose model → Train → Chat → Download adapter</p>', unsafe_allow_html=True)

tab1, tab2, tab3 = st.tabs(["🚀 Train", "💬 Chat", "📥 Export"])

# ── TRAIN TAB ───────────────────────────────────────
with tab1:
    with st.container():
        st.markdown('<div class="card">', unsafe_allow_html=True)
        st.subheader("Train your model")

        col1, col2 = st.columns(2)
        with col1:
            model_name = st.selectbox("Base model", list(MODEL_OPTIONS.keys()), index=0)
        with col2:
            dataset_file = st.file_uploader("Upload your dataset (.txt)", type="txt")

        hf_token = st.text_input(
            "Hugging Face Token (only needed for Gemma)",
            type="password",
            placeholder="hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
            help="Required only if you choose Gemma 2B"
        )

        if st.button("🚀 Start Fine-Tuning", type="primary", use_container_width=True):
            if dataset_file is None:
                st.warning("Please upload a .txt dataset file.")
            else:
                with tempfile.NamedTemporaryFile(delete=False, suffix=".txt") as temp_file:
                    temp_file.write(dataset_file.getvalue())
                    temp_path = temp_file.name

                success, _ = start_training(model_name, temp_path, hf_token)
                os.unlink(temp_path)

                if success:
                    st.balloons()

        st.markdown('</div>', unsafe_allow_html=True)

# ── CHAT TAB ────────────────────────────────────────
with tab2:
    with st.container():
        st.markdown('<div class="card">', unsafe_allow_html=True)
        st.subheader("Chat with your fine-tuned model")

        if st.session_state.finetuned_model is None:
            st.info("👋 Please train a model first in the Train tab.")
        else:
            for msg in st.session_state.chat_history:
                with st.chat_message(msg["role"]):
                    st.markdown(msg["content"])

            user_input = st.chat_input("Ask a question related to your data...")
            if user_input:
                st.session_state.chat_history.append({"role": "user", "content": user_input})
                with st.chat_message("user"):
                    st.markdown(user_input)

                with st.chat_message("assistant"):
                    with st.spinner("Thinking..."):
                        response = chat_response(user_input)
                    st.markdown(response)

                st.session_state.chat_history.append({"role": "assistant", "content": response})

        if st.button("🧹 Clear conversation", use_container_width=True):
            st.session_state.chat_history = []
            st.rerun()

        st.markdown('</div>', unsafe_allow_html=True)

# ── EXPORT TAB ──────────────────────────────────────
with tab3:
    with st.container():
        st.markdown('<div class="card">', unsafe_allow_html=True)
        st.subheader("Export your trained adapter")

        if st.session_state.adapter_path:
            zip_path = export_adapter()
            if zip_path:
                with open(zip_path, "rb") as f:
                    st.download_button(
                        label="📥 Download Adapter (.zip)",
                        data=f,
                        file_name="lora_adapter.zip",
                        mime="application/zip",
                        use_container_width=True
                    )
        else:
            st.info("Train a model first to generate an adapter.")

        st.markdown('</div>', unsafe_allow_html=True)

# ── Footer ──────────────────────────────────────────
st.markdown("""
<div style="text-align:center; color:#6b7280; font-size:0.9rem; margin:2.5rem 0 1rem 0; padding-top:1.2rem; border-top:1px solid #e2e8f0;">
    Training speed depends on dataset size & GPU availability.<br>
    Small models like Qwen 0.5B are fastest and use least memory.
</div>
""", unsafe_allow_html=True)

Writing app.py


In [ ]:
# ngrok + Streamlit launc)

NGROK_AUTH_TOKEN = "3AFlg9Z9NHzr6veHUCHSuNQzSDV_6GGZ6HYQpdmy9cNrLam36"  # https://dashboard.ngrok.com/get-started/your-authtoken

!pip install pyngrok --quiet
from pyngrok import ngrok
import time
import os

os.system("pkill -f ngrok || true")

os.system(f"ngrok authtoken {NGROK_AUTH_TOKEN}")

public_url = ngrok.connect(8012, "http")
print(f"Public Streamlit URL: {public_url}")

!streamlit run app.py --server.port 8012 --server.address 0.0.0.0 --theme.base light &>/dev/null &

Public Streamlit URL: NgrokTunnel: "https://unmilitaristic-alexander-intermetallic.ngrok-free.dev" -> "http://localhost:8012"
